# Join Cleaned Data

**Merges all three cleaned data sources into a single wide CSV for both the training and challenge sets.**
- Input: `hai_cleaned.csv`, `participants_cleaned.csv`, `transcriptomics_cleaned.csv`
- Input: `challenge_hai_cleaned.csv`, `challenge_participants_cleaned.csv`, `challenge_transcriptomics_cleaned.csv`
- Output: `train_combined.csv`, `challenge_combined.csv`

## Design notes

**Reference set:** HAI defines the participant universe for each split. All other tables are left-joined to it, so every participant row is preserved even if they lack transcriptomics data.

**Column order:** `PART_*` demographics → `HAI_*` titers → `TRAN_PC*` transcriptomics PCs.

**Challenge data:** Only Day 0 HAI titers are available (17 strains). Transcriptomics PCs are projected into the same space as training using the training-fit PCA model.

In [1]:
CLEANED_DIR = '../cleaned_data'

In [2]:
import os

import pandas as pd

from utils import peek

## Training Data

### Load cleaned data

In [3]:
hai_df = pd.read_csv(CLEANED_DIR + '/hai_cleaned.csv', low_memory=False)
hai_participants = set(hai_df['participant_id'].unique())
print(f'HAI shape: {hai_df.shape}')
print(f'Unique participants: {len(hai_participants)}')
peek(hai_df)

HAI shape: (3757, 196)
Unique participants: 3757


,participant_id,HAI_Anc B/Lee/1940_d0,HAI_Anc B/Lee/1940_d28,HAI_Anc B/Lee/1940_d365,HAI_Anc B/Maryland/1959_d0,HAI_Anc B/Maryland/1959_d28,HAI_Anc B/Singapore/1964_d0,HAI_Anc B/Singapore/1964_d28,HAI_H1N1 A/Beijing/262/1995_d0,HAI_H1N1 A/Beijing/262/1995_d28,HAI_H1N1 A/Beijing/262/1995_d365,HAI_H1N1 A/Brazil/11/1978_d0,HAI_H1N1 A/Brazil/11/1978_d28,HAI_H1N1 A/Brazil/11/1978_d365,HAI_H1N1 A/Brisbane/2/2018_d0,HAI_H1N1 A/Brisbane/2/2018_d28,HAI_H1N1 A/Brisbane/2/2018_d365,HAI_H1N1 A/Brisbane/59/2007_d0,HAI_H1N1 A/Brisbane/59/2007_d28,HAI_H1N1 A/Brisbane/59/2007_d365
0,2016_UGA.ID_001,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.321928,5.321928,NaN,2.321928,2.321928,NaN,NaN,NaN,NaN,3.321928,5.321928,NaN
1,2016_UGA.ID_002,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6.321928,5.321928,NaN,2.321928,2.321928,NaN,NaN,NaN,NaN,5.321928,5.321928,NaN
2,2016_UGA.ID_003,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6.321928,5.321928,NaN,2.321928,2.321928,NaN,NaN,NaN,NaN,3.321928,3.321928,NaN
3,2016_UGA.ID_004,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.321928,5.321928,NaN,2.321928,2.321928,NaN,NaN,NaN,NaN,3.321928,5.321928,NaN
4,2016_UGA.ID_005,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.321928,2.321928,2.321928,2.321928,2.321928,2.321928,NaN,NaN,NaN,2.321928,2.321928,3.321928


In [4]:
participants_df = pd.read_csv(CLEANED_DIR + '/participants_cleaned.csv', low_memory=False)
participants_df = participants_df[participants_df['participant_id'].isin(hai_participants)]
print(f'Participants shape: {participants_df.shape}')
peek(participants_df)

Participants shape: (3757, 4)


,participant_id,PART_biological_sex,PART_arm_name,PART_age
0,SDY269.SUB112836,female,Other,28.0
1,SDY269.SUB112849,female,Other,39.0
2,SDY269.SUB112854,male,Other,46.0
3,SDY269.SUB112860,female,Other,32.0
4,SDY269.SUB112881,female,Other,29.0


In [5]:
transcriptomics_df = pd.read_csv(CLEANED_DIR + '/transcriptomics_cleaned.csv')
print(f'Transcriptomics shape: {transcriptomics_df.shape}')
peek(transcriptomics_df)

Transcriptomics shape: (395, 291)


,participant_id,TRAN_PC1,TRAN_PC2,TRAN_PC3,TRAN_PC4,TRAN_PC5,TRAN_PC6,TRAN_PC7,TRAN_PC8,TRAN_PC9,TRAN_PC10,TRAN_PC11,TRAN_PC12,TRAN_PC13,TRAN_PC14,TRAN_PC15,TRAN_PC16,TRAN_PC17,TRAN_PC18,TRAN_PC19
0,2019_UGA.ID_001,17.963265,10.116922,0.201273,-17.255525,3.202053,-1.094427,-7.256771,-1.322318,-4.153648,-0.613404,-9.614626,0.720992,4.885530,-4.884685,5.501272,7.561164,-1.978462,-7.722734,-4.200289
1,2019_UGA.ID_005,24.355613,-8.863714,-4.527481,-10.857142,-6.061716,0.245362,-3.501424,-0.653249,9.783679,-0.592398,0.533976,1.964981,-2.689167,0.460776,-1.398064,1.393204,0.935714,6.719536,1.318274
2,2019_UGA.ID_008,4.711335,5.099504,-0.158194,-1.316964,5.371218,-2.394113,-7.784587,1.303140,-0.666468,-0.732497,-7.276848,11.206923,4.566052,1.177469,3.608693,-0.861909,-3.937105,-3.548161,1.860893
3,2019_UGA.ID_011,19.741664,-14.623409,-6.113772,7.795168,-0.831130,5.116629,8.460787,2.478725,1.777897,-0.121422,-0.683163,-0.808980,-1.829761,0.306623,4.709386,-1.172098,2.365919,0.285702,-2.150047
4,2019_UGA.ID_014,26.720715,-15.366108,-2.405799,2.701028,-4.162444,1.662702,-2.216251,1.304555,8.861099,-0.719469,-1.287278,2.894646,-3.274467,1.600970,0.040856,-1.553626,1.243765,1.355677,-0.384338


### Merge

In [6]:
merged = participants_df.merge(hai_df, on='participant_id', how='left')
merged = merged.merge(transcriptomics_df, on='participant_id', how='left')
print(f'Merged shape: {merged.shape}')
peek(merged)

Merged shape: (3757, 489)


,participant_id,PART_biological_sex,PART_arm_name,PART_age,HAI_Anc B/Lee/1940_d0,HAI_Anc B/Lee/1940_d28,HAI_Anc B/Lee/1940_d365,HAI_Anc B/Maryland/1959_d0,HAI_Anc B/Maryland/1959_d28,HAI_Anc B/Singapore/1964_d0,HAI_Anc B/Singapore/1964_d28,HAI_H1N1 A/Beijing/262/1995_d0,HAI_H1N1 A/Beijing/262/1995_d28,HAI_H1N1 A/Beijing/262/1995_d365,HAI_H1N1 A/Brazil/11/1978_d0,HAI_H1N1 A/Brazil/11/1978_d28,HAI_H1N1 A/Brazil/11/1978_d365,HAI_H1N1 A/Brisbane/2/2018_d0,HAI_H1N1 A/Brisbane/2/2018_d28,HAI_H1N1 A/Brisbane/2/2018_d365
0,SDY269.SUB112836,female,Other,28.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,SDY269.SUB112849,female,Other,39.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,SDY269.SUB112854,male,Other,46.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,SDY269.SUB112860,female,Other,32.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,SDY269.SUB112881,female,Other,29.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [7]:
os.makedirs(CLEANED_DIR, exist_ok=True)
merged.to_csv(CLEANED_DIR + '/train_combined.csv', index=False)
print(f'Saved to {CLEANED_DIR}/train_combined.csv')

Saved to ../cleaned_data/train_combined.csv


## Challenge Data

### Load cleaned data

In [8]:
chal_hai_df = pd.read_csv(CLEANED_DIR + '/challenge_hai_cleaned.csv', low_memory=False)
chal_participants = set(chal_hai_df['participant_id'].unique())
print(f'Challenge HAI shape: {chal_hai_df.shape}')
print(f'Unique participants: {len(chal_participants)}')
peek(chal_hai_df)

Challenge HAI shape: (40, 18)
Unique participants: 40


,participant_id,HAI_H1N1 A/Brisbane/2/2018_d0,HAI_H1N1 A/California/7/2009_d0,HAI_H1N1 A/Guangdong-Maonan/SWL1536/2019_d0,HAI_H1N1 A/Victoria/2570/2019_d0,HAI_H1N1 A/Victoria/4897/2022_d0,HAI_H3N2 A/Darwin/9/2021_d0,HAI_H3N2 A/Hong Kong/2671/2019_d0,HAI_H3N2 A/Hong Kong/4801/2014_d0,HAI_H3N2 A/Kansas/14/2017_d0,HAI_H3N2 A/Massachusetts/18/2022_d0,HAI_H3N2 A/Singapore/INFIMH-160019/2016_d0,HAI_H3N2 A/South Australia/34/2019_d0,HAI_H3N2 A/Tasmania/503/2020_d0,HAI_Vic B/Austria/1359417/2021_d0,HAI_Vic B/Colorado/6/2017_d0,HAI_Vic B/Washington/2/2019_d0,HAI_Yam B/Phuket/3073/2013_d0
0,2024_UGA.ID_077,2.321928,4.321928,2.321928,2.321928,2.321928,2.321928,4.321928,3.321928,3.321928,2.321928,3.321928,6.321928,3.321928,4.321928,2.321928,2.321928,4.321928
1,2024_UGA.ID_086,8.321928,9.321928,9.321928,6.321928,4.321928,2.321928,2.321928,2.321928,2.321928,2.321928,2.321928,4.321928,3.321928,5.321928,5.321928,4.321928,5.321928
2,2024_UGA.ID_128,6.321928,7.321928,6.321928,6.321928,5.321928,6.321928,7.321928,7.321928,7.321928,7.321928,7.321928,8.321928,7.321928,3.321928,3.321928,3.321928,5.321928
3,2024_UGA.ID_170,3.321928,3.321928,3.321928,5.321928,2.321928,3.321928,5.321928,3.321928,4.321928,7.321928,4.321928,5.321928,6.321928,3.321928,2.321928,2.321928,3.321928
4,2024_UGA.ID_179,5.321928,3.321928,2.321928,6.321928,2.321928,4.321928,3.321928,2.321928,6.321928,3.321928,2.321928,4.321928,3.321928,5.321928,4.321928,4.321928,7.321928


In [9]:
chal_participants_df = pd.read_csv(CLEANED_DIR + '/challenge_participants_cleaned.csv', low_memory=False)
chal_participants_df = chal_participants_df[chal_participants_df['participant_id'].isin(chal_participants)]
print(f'Challenge participants shape: {chal_participants_df.shape}')
peek(chal_participants_df)

Challenge participants shape: (40, 4)


,participant_id,PART_biological_sex,PART_arm_name,PART_age
0,2024_UGA.ID_077,female,High Dose Fluzone,72.0
1,2024_UGA.ID_086,male,High Dose Fluzone,65.0
2,2024_UGA.ID_128,female,Standard Fluzone,43.0
3,2024_UGA.ID_170,female,High Dose Fluzone,74.0
4,2024_UGA.ID_179,female,High Dose Fluzone,70.0


In [10]:
chal_transcriptomics_df = pd.read_csv(CLEANED_DIR + '/challenge_transcriptomics_cleaned.csv')
print(f'Challenge transcriptomics shape: {chal_transcriptomics_df.shape}')
peek(chal_transcriptomics_df)

Challenge transcriptomics shape: (40, 291)


,participant_id,TRAN_PC1,TRAN_PC2,TRAN_PC3,TRAN_PC4,TRAN_PC5,TRAN_PC6,TRAN_PC7,TRAN_PC8,TRAN_PC9,TRAN_PC10,TRAN_PC11,TRAN_PC12,TRAN_PC13,TRAN_PC14,TRAN_PC15,TRAN_PC16,TRAN_PC17,TRAN_PC18,TRAN_PC19
0,2024_UGA.ID_077,-11.011232,0.515703,-1.834634,-2.308493,2.678642,0.563072,-1.961106,0.534324,-1.369662,-0.539273,-7.793562,2.528308,-1.088294,-3.015375,3.250450,0.446284,1.385004,-2.331847,-2.188888
1,2024_UGA.ID_086,-9.813977,-1.369324,-2.334222,3.858163,4.146166,1.935869,3.341677,1.925138,-4.370104,0.031000,-5.220041,-0.863788,1.082808,-0.812443,0.279127,2.010819,-0.232852,2.603671,-0.197856
2,2024_UGA.ID_128,-14.776537,-3.026487,-4.328682,-3.087700,-1.543489,-1.788575,-6.254604,-1.171083,1.635093,0.034767,-6.311059,0.624729,2.822985,-3.242062,1.312404,5.409135,-1.586806,3.565729,0.942392
3,2024_UGA.ID_170,-16.091678,-1.171347,-1.971571,-3.060197,4.830262,0.353944,-1.517147,0.622641,-0.324396,-0.602554,-8.274639,3.565256,-3.676462,-4.387397,3.654042,1.329076,2.274200,-4.384275,-3.755314
4,2024_UGA.ID_179,2.300038,2.438006,-0.378481,1.915821,-3.034490,0.290041,-4.562120,-0.905066,-2.455151,0.988734,-8.794241,1.242418,-0.248117,0.949222,-2.363575,0.773956,0.154369,5.576212,2.701130


### Merge

In [11]:
chal_merged = chal_participants_df.merge(chal_hai_df, on='participant_id', how='left')
chal_merged = chal_merged.merge(chal_transcriptomics_df, on='participant_id', how='left')
print(f'Challenge merged shape: {chal_merged.shape}')
peek(chal_merged)

Challenge merged shape: (40, 311)


,participant_id,PART_biological_sex,PART_arm_name,PART_age,HAI_H1N1 A/Brisbane/2/2018_d0,HAI_H1N1 A/California/7/2009_d0,HAI_H1N1 A/Guangdong-Maonan/SWL1536/2019_d0,HAI_H1N1 A/Victoria/2570/2019_d0,HAI_H1N1 A/Victoria/4897/2022_d0,HAI_H3N2 A/Darwin/9/2021_d0,HAI_H3N2 A/Hong Kong/2671/2019_d0,HAI_H3N2 A/Hong Kong/4801/2014_d0,HAI_H3N2 A/Kansas/14/2017_d0,HAI_H3N2 A/Massachusetts/18/2022_d0,HAI_H3N2 A/Singapore/INFIMH-160019/2016_d0,HAI_H3N2 A/South Australia/34/2019_d0,HAI_H3N2 A/Tasmania/503/2020_d0,HAI_Vic B/Austria/1359417/2021_d0,HAI_Vic B/Colorado/6/2017_d0,HAI_Vic B/Washington/2/2019_d0
0,2024_UGA.ID_077,female,High Dose Fluzone,72.0,2.321928,4.321928,2.321928,2.321928,2.321928,2.321928,4.321928,3.321928,3.321928,2.321928,3.321928,6.321928,3.321928,4.321928,2.321928,2.321928
1,2024_UGA.ID_086,male,High Dose Fluzone,65.0,8.321928,9.321928,9.321928,6.321928,4.321928,2.321928,2.321928,2.321928,2.321928,2.321928,2.321928,4.321928,3.321928,5.321928,5.321928,4.321928
2,2024_UGA.ID_128,female,Standard Fluzone,43.0,6.321928,7.321928,6.321928,6.321928,5.321928,6.321928,7.321928,7.321928,7.321928,7.321928,7.321928,8.321928,7.321928,3.321928,3.321928,3.321928
3,2024_UGA.ID_170,female,High Dose Fluzone,74.0,3.321928,3.321928,3.321928,5.321928,2.321928,3.321928,5.321928,3.321928,4.321928,7.321928,4.321928,5.321928,6.321928,3.321928,2.321928,2.321928
4,2024_UGA.ID_179,female,High Dose Fluzone,70.0,5.321928,3.321928,2.321928,6.321928,2.321928,4.321928,3.321928,2.321928,6.321928,3.321928,2.321928,4.321928,3.321928,5.321928,4.321928,4.321928


In [12]:
os.makedirs(CLEANED_DIR, exist_ok=True)
chal_merged.to_csv(CLEANED_DIR + '/challenge_combined.csv', index=False)
print(f'Saved to {CLEANED_DIR}/challenge_combined.csv')

Saved to ../cleaned_data/challenge_combined.csv
